# 第1章  数、变量与函数 —— 用Python重温初中数学

> **本章目标**：建立数学概念到Python表达的思维反射。理解float的精度陷阱，掌握NumPy数组为什么比Python列表快，学会用matplotlib将任何函数可视化。
> **前置知识**：会用Python写简单循环、定义变量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪 - NumPy + Matplotlib')


---
## 1.1  整数、浮点数与布尔值 —— 计算机里的数不是数学里的数

打开任何一个Python环境，输入 0.1+0.2，你会看到0.30000000000000004而不是0.3。这不是Python的bug——这是所有计算机共享的基因缺陷：浮点数用二进制近似小数。

📐 **IEEE 754浮点数**：用1比特符号位+11比特指数+52比特尾数（共64bits）近似存储实数。约15-17位十进制有效数字。

🔗 **AI连接**：混合精度训练（第20章）、softmax溢出（第22章）都源于浮点数的物理边界。

In [1]:
import math

# 基本类型
a, b, c = 42, 3.14159, True
print(f'a={a}, type={type(a).__name__}')
print(f'b={b}, type={type(b).__name__}')
print(f'c={c}, type={type(c).__name__}')
print(f'True + True = {True + True}')

# 浮点精度陷阱
x = 0.1 + 0.2
print(f'\n0.1 + 0.2 = {x}')
print(f'0.1 + 0.2 == 0.3 ? {x == 0.3}')
print(f'实际差值: {x - 0.3:.2e}')

# 正确比较：容差法
EPS = 1e-10
print(f'\n容差法判断: {abs(x - 0.3) < EPS}')
print(f'math.isclose: {math.isclose(0.1 + 0.2, 0.3)}')

# 浮点数的极限
print(f'\n1.0 + 1e-16 == 1.0 ? {1.0 + 1e-16 == 1.0}')
print(f'1.0 + 1e-15 == 1.0 ? {1.0 + 1e-15 == 1.0}')
big = 2 ** 1000
print(f'\n2^1000 = {big}')
print(f"2^1000 的位数: {len(str(big))}")


a=42, type=int
b=3.14159, type=float
c=True, type=bool
True + True = 2

0.1 + 0.2 = 0.30000000000000004
0.1 + 0.2 == 0.3 ? False
实际差值: 5.55e-17

容差法判断: True
math.isclose: True

1.0 + 1e-16 == 1.0 ? True
1.0 + 1e-15 == 1.0 ? False

2^1000 = 10715086071862673209484250490600018105614048117055336074437503883703510511249361224931983788156958581275946729175531468251871452856923140435984577574698574803934567774824230985421074605062371141877954182153046474983581941267398767559165543946077062914571196477686542167660429831652624386837205668069376
2^1000 的位数: 302


---
## 1.2  从列表到NumPy数组 —— AI为什么不用Python列表？

Python列表是指针数组：每个元素是指向分散内存中对象的指针。灵活但慢。NumPy数组ndarray是同类型、连续内存——所有元素紧挨排列，CPU可批量加载、SIMD并行处理。

📐 **NumPy数组（ndarray）**：同类型元素连续存储于内存的多维数组。运算在C层面执行，Python只做调度。

🔗 **AI连接**：PyTorch的torch.Tensor共享同一设计哲学——连续内存、同类型、向量化运算。

In [2]:
import numpy as np
import time

py_list = list(range(1_000_000))
np_arr = np.arange(1_000_000, dtype=np.int64)
print(f'Python list memory: {py_list.__sizeof__() / 1e6:.1f} MB')
print(f'NumPy array memory: {np_arr.nbytes / 1e6:.1f} MB')

t0 = time.perf_counter()
squared_list = [x ** 2 for x in py_list]
t1 = time.perf_counter()

t2 = time.perf_counter()
squared_arr = np_arr ** 2
t3 = time.perf_counter()

print(f'\nPython list: {t1 - t0:.4f}s')
print(f'NumPy array: {t3 - t2:.4f}s')
print(f'NumPy is {(t1-t0)/(t3-t2):.0f}x faster')

arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print(f'\narr={arr}')
print(f'arr+10={arr+10}')
print(f'arr[arr>3]={arr[arr>3]}')
print(f'sum={arr.sum()}, mean={arr.mean():.2f}')


Python list memory: 8.0 MB
NumPy array memory: 8.0 MB

Python list: 0.0899s
NumPy array: 0.0024s
NumPy is 37x faster

arr=[3 1 4 1 5 9 2 6]
arr+10=[13 11 14 11 15 19 12 16]
arr[arr>3]=[4 5 9 6]
sum=31, mean=3.88


---
## 1.3  函数就是输入到输出的盒子 —— 数学与代码的桥梁

数学课本上写f(x)=x^2，Python里写def f(x):return x**2。两者本质相同：函数=输入到输出的映射规则。

神经网络=f(x;theta)=一个将输入映射到输出概率的巨大函数。

📐 **np.linspace**：在区间内均匀采样n个点——将连续函数离散化以绘图的标配工具。

🔗 **AI连接**：这个绘图模板将贯穿全书。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_function(f, x_range=(-3, 3), n_points=200, title='f(x)'):
    x = np.linspace(x_range[0], x_range[1], n_points)
    y = f(x)
    plt.figure(figsize=(7, 4))
    plt.plot(x, y, 'b-', linewidth=2)
    plt.axhline(y=0, color='gray', linewidth=0.5)
    plt.axvline(x=0, color='gray', linewidth=0.5)
    plt.xlabel('x'); plt.ylabel('f(x)')
    plt.title(title); plt.grid(alpha=0.3); plt.show()

# y = 2x + 3
plot_function(lambda x: 2*x + 3, x_range=(-4, 2), title='f(x) = 2x + 3')
print('直线——斜率恒定')


In [ ]:
# y = x^2
plot_function(lambda x: x**2, x_range=(-3, 3), title='f(x) = x^2')
print('抛物线——斜率在变')


In [ ]:
# 对比
x = np.linspace(-3, 3, 200)
plt.figure(figsize=(7, 4))
plt.plot(x, 2*x + 3, 'b-', linewidth=2, label='y=2x+3')
plt.plot(x, x**2, 'r-', linewidth=2, label='y=x^2')
plt.axhline(y=0, color='gray', linewidth=0.5)
plt.axvline(x=0, color='gray', linewidth=0.5)
plt.xlabel('x'); plt.ylabel('y')
plt.title('直线 vs 抛物线')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('直线的倾斜度处处相同，抛物线的倾斜度随x而变——这就是导数的核心直觉')


---
## 习题

> 在下方新建代码单元格作答。

1. （概念）0.1+0.2==0.3为什么返回False？
2. （概念）为什么NumPy数组运算远快于Python列表？
3. （概念）np.linspace(-3,3,200)做了什么？如果n_points=5会怎样？
4. （代码）计算0.1+0.1+0.1-0.3，用math.isclose验证。
5. （代码）用plot_function在同一图上画x^3、sigmoid、ReLU，x in [-2,2]

---

> **章末钩子**：你现在会用Python表达数、数组、函数了。但函数值在变化这件事本身也可以被量化——比如抛物线y=x^2在x=2处的陡峭程度是多少？
> 预览：下一章——**变化的直觉**，用割线逼近切线。

> **提示**：完成后运行 Kernel -> Restart & Run All 验证。